In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

C:\Users\Admin\AppData\Local\Temp\ipykernel_33764\1856170868.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### Load and Split the Dataset

In [2]:
loader=TextLoader("rag_query_enhancement_dataset_211_lines_no_numbers.txt")
raw_docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(raw_docs)

In [3]:
chunks

[Document(metadata={'source': 'rag_query_enhancement_dataset_211_lines_no_numbers.txt'}, page_content='Retrieval Augmented Generation, commonly called RAG, combines retrieval with language generation.\nA RAG system first searches external knowledge before asking a language model to answer.\nThe external knowledge can contain documents, manuals, websites, databases, or internal company information.'),
 Document(metadata={'source': 'rag_query_enhancement_dataset_211_lines_no_numbers.txt'}, page_content='RAG reduces dependence on facts stored inside the language model parameters.\nIt can also provide current information without completely retraining the language model.\nA typical RAG pipeline contains ingestion, chunking, embedding, retrieval, prompting, and generation.'),
 Document(metadata={'source': 'rag_query_enhancement_dataset_211_lines_no_numbers.txt'}, page_content='During ingestion, source documents are loaded and converted into machine-readable text.\nChunking divides long docum

### Vectore Store

In [4]:
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vstore=Chroma.from_documents(chunks,embedding_model)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### MMR Retriever

In [5]:
retriever=vstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001369F3ACB50>, search_type='mmr', search_kwargs={'k': 5})

### LLM and Prompt

In [17]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen/qwen3-vl-4b",
    base_url="http://127.0.0.1:1234/v1",
    api_key="dummy",
    temperature=0.7
    

)

### Query Expansion

In [18]:
query_expansion=PromptTemplate.from_template("""

You are a helpful assistant.Expand the following query to improve document retrieval by adding relevant synonyms , technical terms , and useful context.

Orignal query:"{query}"

Expanded query:

""")

query_expansion_chain= query_expansion | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\n\nYou are a helpful assistant.Expand the following query to improve document retrieval by adding relevant synonyms , technical terms , and useful context.\n\nOrignal query:"{query}"\n\nExpanded query:\n\n')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14', 'langchain-openai': '1.4.0'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000013709BE07D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000013709BBB350>, root_client=<openai.OpenAI object at 0x0000013709BE0E10>, root_async_client=<openai.AsyncOpenAI object at 0x0000013709BB8610>, model_name='qwen/qwen3-vl-4b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://127.0.0.1:1234/v1', openai_proxy=None, stream_chunk_timeout=120.0)
| StrOutputParser()

In [19]:
query_expansion_chain.invoke({"query":"RAG Memory"})

'Expanded Query:\n\n“RAG Memory” — Retrieval-Augmented Generation Memory, also known as Retrieval-Augmented Generation Knowledge Base, Document Memory, or Contextual Knowledge Store. This refers to the mechanism in large language models (LLMs) that integrates external, retrievable information sources (such as documents, databases, or web pages) into the generation process to enhance factual accuracy, relevance, and context-awareness. Synonyms and related terms include: Knowledge Retrieval Module, Contextual Memory Embedding, Document Embedding Store, Vector Database Index, Semantic Search Index, Retrieval Layer, Knowledge Base Layer, External Knowledge Integration, Document Retrieval Engine, Passage Retrieval System, and Contextual Memory Buffer. Technical context: RAG Memory typically involves vectorization of documents using embeddings (e.g., BERT, Sentence-BERT, or OpenAI embeddings), similarity search (e.g., FAISS, Annoy, Milvus), and contextual fusion with the LLM’s internal reaso

### RAG ans prompt

In [20]:
answer_prompt=PromptTemplate.from_template("""

Answer the question based on the context below.

Context:
{context}

Question:{input}

""")


document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

### FUll RAG Pipeline with expansion

In [21]:
rag_pipeline=(
    RunnableMap({
        "input":lambda x:x["input"],
        "context":lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain

)

In [22]:
query={"input":"what type of memory does RAG support?"}
print(query_expansion_chain.invoke({"query":query}))
response=rag_pipeline.invoke(query)
print("ANSWER:\n", response)

Expanded query:

"What types of memory are supported by Retrieval-Augmented Generation (RAG) systems? — including but not limited to: short-term memory (e.g., context window memory), long-term memory (e.g., document store, knowledge base, vector database), persistent memory (e.g., Redis, MongoDB, Pinecone), and memory buffers (e.g., caching mechanisms, session memory). Also, include technical terms such as: vector retrieval, semantic memory, retrieval memory, context memory, document embeddings, and memory-augmented neural networks. Context: RAG systems typically combine a retrieval module (e.g., dense vector search, BM25, hybrid search) with a generation module (e.g., transformer models like GPT, LLaMA) to dynamically access external knowledge during inference. Consider use cases such as enterprise knowledge bases, research assistants, or conversational AI systems that require real-time access to curated documents or databases. Include relevant synonyms: ‘memory mechanisms’, ‘knowledg